In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.visualization
import pandas
import named_arrays as na
import esis

In [ ]:
instrument = esis.flights.f2.optics.design_single(num_distribution=0)
grating = instrument.grating

In [ ]:
surface = grating.surface

y_skinny = -(grating.halfwidth_inner + grating.width_border_inner)
y_fat = grating.halfwidth_outer + grating.width_border

with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    ax.set_aspect("equal")
    surface.aperture_mechanical.plot(
        ax=ax,
        components=("y", "x"),
        color="black",
    )
    surface.aperture.plot(
        ax=ax,
        components=("y", "x"),
        color="tab:blue",
    )
    ax.scatter(0 * u.mm, 0 * u.mm, color="black", zorder=3)
    ax.text(x=0.5, y=0.8, s="vertex")
    for y_groove in np.linspace(-6, 6, 7):
        ax.plot(
            [-3, 3] * u.mm,
            [y_groove, y_groove] * u.mm,
            color="tab:blue",
            alpha=0.3,
        )
    ax.text(
        x=0,
        y=y_skinny.to_value(u.mm) - 1,
        s="skinny end",
        ha="center",
        va="top",
    )
    ax.text(
        x=0,
        y=y_fat.to_value(u.mm) + 1,
        s="fat end",
        ha="center",
        va="bottom",
    )
    ax.text(
        x=6,
        y=0,
        s="grooves",
        ha="center",
        va="center",
        color="tab:blue",
    )
    ax.set_xlim(-16, 16)
    ax.set_ylim(-17, 14)
    ax.set_xlabel(f"$x$ ({ax.get_xlabel()})")
    ax.set_ylabel(f"$y$ ({ax.get_ylabel()})")
    ax.set_title("view of the optical face, mechanical (black) and clear (blue)")

In [ ]:
grating.rulings.spacing

In [ ]:
y = na.linspace(y_skinny, y_fat, axis="y", num=25)

position = na.Cartesian3dVectorArray(x=y, y=0 * u.mm, z=0 * u.mm)
normal = na.Cartesian3dVectorArray(0, 0, -1)

spacing = grating.rulings.spacing(position, normal).length

print(
    pandas.DataFrame(
        {
            "y (mm)": y.ndarray.to_value(u.mm),
            "spacing (um)": spacing.ndarray.to_value(u.um),
            "density (1/mm)": (1 / spacing).ndarray.to_value(1 / u.mm),
        }
    )
)

In [ ]:
wavelength_center = (
    esis.flights.f2.wavelength_Ne_VII + esis.flights.f2.wavelength_Si_XII
) / 2

ratio = esis.flights.f2.wavelength_HeNe / wavelength_center
ratio = ratio.to(u.dimensionless_unscaled)
ratio

In [ ]:
instrument_visible = esis.flights.f2.optics.design_visible(num_distribution=0)
grating_visible = instrument_visible.grating

grating_visible.rulings.spacing

In [ ]:
spacing_visible = grating_visible.rulings.spacing(position, normal).length

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.plot(y, spacing_visible.to(u.um), ax=ax, color="tab:blue")
    ax.axvline(
        -grating.halfwidth_inner.to_value(u.mm),
        color="gray",
        linestyle="--",
    )
    ax.axvline(
        grating.halfwidth_outer.to_value(u.mm),
        color="gray",
        linestyle="--",
        label="clear aperture",
    )
    ax.text(0.01, 0.02, "skinny end", transform=ax.transAxes, ha="left", va="bottom")
    ax.text(0.99, 0.02, "fat end", transform=ax.transAxes, ha="right", va="bottom")
    ax.set_xlabel(f"$y$ ({ax.get_xlabel()})")
    ax.set_ylabel(f"ruling spacing ({ax.get_ylabel()})")
    ax.legend()

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.plot(y, (1 / spacing_visible).to(1 / u.mm), ax=ax, color="tab:blue")
    ax.axvline(
        -grating.halfwidth_inner.to_value(u.mm),
        color="gray",
        linestyle="--",
    )
    ax.axvline(
        grating.halfwidth_outer.to_value(u.mm),
        color="gray",
        linestyle="--",
        label="clear aperture",
    )
    ax.text(0.01, 0.98, "skinny end", transform=ax.transAxes, ha="left", va="top")
    ax.text(0.99, 0.98, "fat end", transform=ax.transAxes, ha="right", va="top")
    ax.set_xlabel(f"$y$ ({ax.get_xlabel()})")
    ax.set_ylabel(f"ruling density ({ax.get_ylabel()})")
    ax.legend()

In [ ]:
print(
    pandas.DataFrame(
        {
            "y (mm)": y.ndarray.to_value(u.mm),
            "spacing (um)": spacing_visible.ndarray.to_value(u.um),
            "density (1/mm)": (1 / spacing_visible).ndarray.to_value(1 / u.mm),
        }
    )
)

In [ ]:
instrument_visible.field.num = 3
instrument_visible.pupil.num = 3

with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    instrument_visible.system.plot(
        components=("z", "x"),
        color="black",
        kwargs_rays=dict(
            color="tab:red",
        ),
    );

In [ ]:
rays_visible = instrument_visible.system.rayfunction().outputs

where = rays_visible.unvignetted
position_visible = (rays_visible.position.x * where).sum() / where.sum()
position_visible.ndarray.to(u.mm)

In [ ]:
instrument_euv = esis.flights.f2.optics.design(num_distribution=0)
instrument_euv.field.num = 3
instrument_euv.pupil.num = 3

rays_euv = instrument_euv.system.rayfunction().outputs

axis = tuple(a for a in rays_euv.position.x.shape if a != "wavelength")

where = rays_euv.unvignetted
position_euv = (rays_euv.position.x * where).sum(axis=axis) / where.sum(axis=axis)
position_euv.ndarray.to(u.mm)

In [ ]:
position_difference = position_visible - position_euv.mean(axis="wavelength")
position_difference.ndarray.to(u.um)